# Flight Delay ML Platform — Seasonally Matched Historical Features

## 1. Research Objective

Investigate whether historical delay statistics from the
same calendar month of the previous year improve flight
delay prediction.

## Motivation

Previous historical features were constructed from the
three most recently available reporting months under an
assumed 120-day publication lag.

The resulting historical rates exhibited substantial
seasonal mismatch and limited temporal generalization.

## Feature Design

- Previous-year same-month origin departure delay rate
- Previous-year same-month origin outbound arrival delay rate
- Previous-year same-month airline-origin departure delay rate
- Previous-year same-month airline-origin arrival delay rate

## Leakage Control

- Historical source months must precede prediction cutoffs
  under the existing research-proxy availability policy.
- No target-month outcomes are used to construct features.
- The original temporal data splits remain unchanged.
- Historical source availability is assumed, not verified.

## Hypothesis

Seasonally matched historical information may provide more
stable predictive signals than seasonally mismatched
historical information.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from flight_delay.data.manifest import sha256_file
from flight_delay.data.storage import load_parquet

from flight_delay.features.availability import (
    build_proxy_availability_calendar,
)

from flight_delay.features.prediction_time import (
    build_prediction_timestamps,
)

from flight_delay.features.historical_aggregation import (
    aggregate_monthly_delay_counts,
)

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").is_file()

PROCESSED_DIR = ROOT / "data/processed"
MANIFEST_DIR = ROOT / "data/interim/manifests"

LAG_DAYS = 120
MIN_AIRLINE_FLIGHTS = 100

print("Project root:", ROOT)
print("Proxy lag:", LAG_DAYS)
print("Minimum airline-origin flights:", MIN_AIRLINE_FLIGHTS)

In [ ]:
target_months = pd.period_range(
    "2023-01",
    "2024-08",
    freq="M",
)

seasonal_calendar = pd.DataFrame({
    "target_month": target_months.astype(str),
    "source_month": (target_months - 12).astype(str),
})

source_calendar = build_proxy_availability_calendar(
    seasonal_calendar["source_month"].tolist(),
    lag_days=LAG_DAYS,
)

seasonal_calendar = seasonal_calendar.merge(
    source_calendar[
        [
            "report_month",
            "assumed_available_at_utc",
            "availability_policy",
            "availability_evidence",
        ]
    ],
    left_on="source_month",
    right_on="report_month",
    validate="one_to_one",
).drop(columns="report_month")

assert len(seasonal_calendar) == 20

assert seasonal_calendar["source_month"].nunique() == 20

for month in seasonal_calendar["source_month"]:

    path = MANIFEST_DIR / f"{month}.json"

    assert path.is_file(), f"Missing manifest: {month}"

print("=== Seasonal Historical Source Calendar ===")

display(seasonal_calendar)

print("\nSeasonal source calendar passed!")

In [ ]:
def load_verified_month(month):

    with (MANIFEST_DIR / f"{month}.json").open(
        encoding="utf-8"
    ) as f:
        manifest = json.load(f)

    assert manifest["period"] == month

    path = (
        PROCESSED_DIR
        / manifest["processed"]["filename"]
    )

    assert sha256_file(path) == (
        manifest["processed"]["sha256"]
    )

    data = load_parquet(path)

    assert len(data) == manifest["processed"]["row_count"]

    assert (
        pd.to_datetime(data["FlightDate"])
        .dt.to_period("M")
        .astype(str)
        .eq(month)
        .all()
    )

    return data

In [ ]:
PILOT_MONTH = "2024-07"

source_entry = seasonal_calendar.loc[
    seasonal_calendar["target_month"].eq(PILOT_MONTH)
].iloc[0]

SOURCE_MONTH = source_entry["source_month"]

pilot_target = load_verified_month(PILOT_MONTH)
pilot_source = load_verified_month(SOURCE_MONTH)

prediction_times = build_prediction_timestamps(
    pilot_target
)

resolved = prediction_times["timestamp_status"].eq("ok")

available_at = source_entry["assumed_available_at_utc"]

assert resolved.any()

assert (
    prediction_times.loc[resolved, "prediction_at_utc"]
    >= available_at
).all()

print("=== Seasonal History Eligibility ===")

print("Target month:", PILOT_MONTH)
print("Source month:", SOURCE_MONTH)
print("Source records:", len(pilot_source))
print("Target records:", len(pilot_target))
print("Assumed source availability:", available_at)

print(
    "Earliest prediction cutoff:",
    prediction_times["prediction_at_utc"].min(),
)

print("Unresolved timestamps:", int((~resolved).sum()))

print("\nSeasonal history eligibility passed!")

In [ ]:
origin_stats, airline_stats = (
    aggregate_monthly_delay_counts(
        pilot_source,
        report_month=SOURCE_MONTH,
    )
)

origin_stats["origin_dep_delay_rate_ly"] = (
    origin_stats["dep_delay_count"]
    / origin_stats["flight_count"]
)

origin_stats["origin_outbound_arr_delay_rate_ly"] = (
    origin_stats["arr_delay_count"]
    / origin_stats["flight_count"]
)

airline_stats["airline_origin_dep_delay_rate_ly"] = (
    airline_stats["dep_delay_count"]
    / airline_stats["flight_count"]
)

airline_stats["airline_origin_arr_delay_rate_ly"] = (
    airline_stats["arr_delay_count"]
    / airline_stats["flight_count"]
)

print("=== Seasonal Origin Statistics ===")

display(origin_stats)

print("\n=== Seasonal Airline-Origin Statistics ===")

display(airline_stats.head(15))

In [ ]:
SEASONAL_FEATURE_COLUMNS = [
    "origin_dep_delay_rate_ly",
    "origin_outbound_arr_delay_rate_ly",
    "airline_origin_dep_delay_rate_ly",
    "airline_origin_arr_delay_rate_ly",
]

origin_lookup = origin_stats[
    ["Origin"] + SEASONAL_FEATURE_COLUMNS[:2]
]

airline_lookup = airline_stats[
    ["Origin", "Reporting_Airline", "flight_count"]
    + SEASONAL_FEATURE_COLUMNS[2:]
].rename(
    columns={"flight_count": "history_flight_count"}
)

target_keys = pilot_target[
    ["Origin", "Reporting_Airline"]
].copy()

target_keys["_row_id"] = np.arange(len(target_keys))

seasonal_features = (
    target_keys
    .merge(
        origin_lookup,
        on="Origin",
        how="left",
        validate="many_to_one",
    )
    .merge(
        airline_lookup,
        on=["Origin", "Reporting_Airline"],
        how="left",
        validate="many_to_one",
    )
    .sort_values("_row_id")
    .reset_index(drop=True)
)

np.testing.assert_array_equal(
    seasonal_features["_row_id"].to_numpy(),
    np.arange(len(pilot_target)),
)

assert len(seasonal_features) == len(pilot_target)

unknown = seasonal_features[
    "history_flight_count"
].isna()

low_sample = (
    seasonal_features["history_flight_count"]
    .lt(MIN_AIRLINE_FLIGHTS)
    .fillna(False)
)

fallback = unknown | low_sample

seasonal_features["history_status"] = "direct"

seasonal_features.loc[
    low_sample, "history_status"
] = "fallback_low_sample"

seasonal_features.loc[
    unknown, "history_status"
] = "fallback_unknown_group"

# Fall back to origin-level statistics.
for airline_column, origin_column in zip(
    SEASONAL_FEATURE_COLUMNS[2:],
    SEASONAL_FEATURE_COLUMNS[:2],
):
    seasonal_features.loc[
        fallback, airline_column
    ] = seasonal_features.loc[
        fallback, origin_column
    ]

# Do not construct history for unresolved cutoffs.
seasonal_features.loc[
    ~resolved.to_numpy(),
    SEASONAL_FEATURE_COLUMNS,
] = np.nan

seasonal_features.loc[
    ~resolved.to_numpy(),
    "history_status",
] = "history_unavailable"

features = seasonal_features[
    SEASONAL_FEATURE_COLUMNS
]

valid_features = features.loc[resolved.to_numpy()]

assert valid_features.notna().all().all()

assert valid_features.ge(0).all().all()
assert valid_features.le(1).all().all()

print("=== Seasonal Historical Features ===")

display(seasonal_features.head(10))

print("\nFallback distribution:")

display(
    seasonal_features["history_status"].value_counts()
)

print("\nFeature statistics:")

display(features.describe())

print("\nSeasonal historical feature pilot passed!")

In [ ]:
seasonal_diagnostics = pd.DataFrame([
    {
        "Task": "Departure",
        "ActualDelayRate": pilot_target["DepDel15"].mean(),
        "PreviousV2OriginRate": 0.195015,
        "PreviousV2AirlineRate": 0.193236,
        "SeasonalOriginRate": features[
            "origin_dep_delay_rate_ly"
        ].mean(),
        "SeasonalAirlineRate": features[
            "airline_origin_dep_delay_rate_ly"
        ].mean(),
    },
    {
        "Task": "Arrival",
        "ActualDelayRate": pilot_target["ArrDel15"].mean(),
        "PreviousV2OriginRate": 0.199631,
        "PreviousV2AirlineRate": 0.198340,
        "SeasonalOriginRate": features[
            "origin_outbound_arr_delay_rate_ly"
        ].mean(),
        "SeasonalAirlineRate": features[
            "airline_origin_arr_delay_rate_ly"
        ].mean(),
    },
])

for column in [
    "PreviousV2OriginRate",
    "PreviousV2AirlineRate",
    "SeasonalOriginRate",
    "SeasonalAirlineRate",
]:

    seasonal_diagnostics[f"{column}_AbsGap"] = (
        seasonal_diagnostics[column]
        - seasonal_diagnostics["ActualDelayRate"]
    ).abs()

display(seasonal_diagnostics.round(6))

## 2. Modular Seasonal Historical Features

Implement a reusable seasonal history feature module.

Verify that the modular implementation reproduces the
previously validated July 2024 pilot results.

The availability policy, historical aggregation levels,
and fallback strategy remain unchanged.

In [ ]:
from flight_delay.features.seasonal_history import (
    SEASONAL_FEATURE_COLUMNS,
    build_seasonal_history_features,
)

july_features = build_seasonal_history_features(
    flights=pilot_target,
    source=pilot_source,
    target_month="2024-07",
    source_month="2023-07",
    lag_days=LAG_DAYS,
    min_airline_flights=MIN_AIRLINE_FLIGHTS,
)

pd.testing.assert_frame_equal(
    july_features[
        list(SEASONAL_FEATURE_COLUMNS)
    ].reset_index(drop=True),
    seasonal_features[
        list(SEASONAL_FEATURE_COLUMNS)
    ].reset_index(drop=True),
    check_dtype=False,
)

assert july_features["history_status"].tolist() == (
    seasonal_features["history_status"].tolist()
)

assert july_features.index.equals(pilot_target.index)

print("Feature matrix:", july_features.shape)

display(
    july_features["history_status"].value_counts()
)

print("\nJuly seasonal regression verification passed!")

## 3. Full Seasonal Historical Feature Generation

Construct seasonally matched historical features for
all training and development validation flights.

Each target month is matched exclusively to the same
calendar month of the preceding year.

Verify source integrity, temporal eligibility, feature
coverage, and flight identity alignment.

Unresolved prediction timestamps remain missing.

The original modeling cohort is preserved.

In [ ]:
import time

IDENTITY_COLUMNS = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

seasonal_features_by_month = {}
seasonal_audit_records = []
monthly_diagnostics_records = []

start = time.perf_counter()

for entry in seasonal_calendar.itertuples(index=False):

    target_month = entry.target_month
    source_month = entry.source_month

    source = load_verified_month(source_month)
    target = load_verified_month(target_month)

    generated = build_seasonal_history_features(
        flights=target,
        source=source,
        target_month=target_month,
        source_month=source_month,
        lag_days=LAG_DAYS,
        min_airline_flights=MIN_AIRLINE_FLIGHTS,
    )

    assert generated.index.equals(target.index)

    model_features = generated[
        list(SEASONAL_FEATURE_COLUMNS)
    ]

    complete = model_features.notna().all(axis=1)

    statuses = generated[
        "history_status"
    ].value_counts()

    assert complete.eq(
        generated["history_status"].ne(
            "history_unavailable"
        )
    ).all()

    saved_frame = pd.concat(
        [
            target[IDENTITY_COLUMNS],
            generated,
        ],
        axis=1,
    )

    assert not saved_frame.duplicated(
        IDENTITY_COLUMNS
    ).any()

    assert not saved_frame[
        IDENTITY_COLUMNS
    ].isna().any().any()

    seasonal_features_by_month[
        target_month
    ] = saved_frame

    seasonal_audit_records.append({
        "Month": target_month,
        "SourceMonth": source_month,
        "SourceRecords": len(source),
        "TargetRecords": len(target),
        "CompleteFeatures": int(complete.sum()),
        "MissingFeatures": int((~complete).sum()),
        "DirectMatches": int(
            statuses.get("direct", 0)
        ),
        "LowSampleFallback": int(
            statuses.get("fallback_low_sample", 0)
        ),
        "UnknownGroupFallback": int(
            statuses.get("fallback_unknown_group", 0)
        ),
        "UnavailableHistory": int(
            statuses.get("history_unavailable", 0)
        ),
    })

    # Target labels are used for diagnostics only.
    # Compare rates on the same complete-feature cohort.
    for task, label, origin_col, airline_col in [
        (
            "Departure",
            "DepDel15",
            SEASONAL_FEATURE_COLUMNS[0],
            SEASONAL_FEATURE_COLUMNS[2],
        ),
        (
            "Arrival",
            "ArrDel15",
            SEASONAL_FEATURE_COLUMNS[1],
            SEASONAL_FEATURE_COLUMNS[3],
        ),
    ]:
        monthly_diagnostics_records.append({
            "Month": target_month,
            "Task": task,
            "Records": int(complete.sum()),
            "ActualDelayRate": target.loc[
                complete, label
            ].mean(),
            "SeasonalOriginRate": generated.loc[
                complete, origin_col
            ].mean(),
            "SeasonalAirlineRate": generated.loc[
                complete, airline_col
            ].mean(),
        })

    print(
        f"{target_month} | "
        f"source={source_month} | "
        f"records={len(target):,} | "
        f"coverage={complete.mean():.6%}"
    )

    del source, target, generated, saved_frame
    del model_features, complete

seasonal_audit = pd.DataFrame(
    seasonal_audit_records
)

seasonal_monthly_diagnostics = pd.DataFrame(
    monthly_diagnostics_records
)

print(
    "\nGeneration seconds:",
    round(time.perf_counter() - start, 2),
)

display(seasonal_audit)

print("\nFull seasonal feature generation completed!")

In [ ]:
assert len(seasonal_audit) == 20

assert seasonal_audit["TargetRecords"].sum() == 2_498_494

assert (
    seasonal_audit["CompleteFeatures"]
    + seasonal_audit["MissingFeatures"]
).eq(seasonal_audit["TargetRecords"]).all()

assert seasonal_audit[
    [
        "DirectMatches",
        "LowSampleFallback",
        "UnknownGroupFallback",
        "UnavailableHistory",
    ]
].sum(axis=1).eq(
    seasonal_audit["TargetRecords"]
).all()

assert seasonal_audit["MissingFeatures"].eq(
    seasonal_audit["UnavailableHistory"]
).all()

total_records = seasonal_audit["TargetRecords"].sum()

complete_records = seasonal_audit[
    "CompleteFeatures"
].sum()

print("=== Seasonal Historical Feature Summary ===")
print("Months:", len(seasonal_audit))
print("Total target records:", f"{total_records:,}")
print("Complete features:", f"{complete_records:,}")
print("Missing features:", total_records - complete_records)
print("Coverage:", f"{complete_records / total_records:.6%}")

print("\n=== Match Distribution ===")

display(
    seasonal_audit[
        [
            "DirectMatches",
            "LowSampleFallback",
            "UnknownGroupFallback",
            "UnavailableHistory",
        ]
    ].sum()
)

print("\nSeasonal feature coverage audit passed!")

## 4. Seasonal Historical Rate Diagnostics

Compare previous-year same-month historical delay rates
with observed monthly delay rates.

Evaluate whether seasonal matching reduces aggregate
rate mismatch across training and validation periods.

Aggregate rate agreement does not necessarily imply
improved individual-flight discrimination.

In [ ]:
import matplotlib.pyplot as plt

seasonal_monthly_diagnostics["Split"] = np.where(
    seasonal_monthly_diagnostics["Month"].str.startswith(
        "2023"
    ),
    "Training",
    "Validation",
)

for task in ["Departure", "Arrival"]:

    plot_data = (
        seasonal_monthly_diagnostics.loc[
            seasonal_monthly_diagnostics["Task"].eq(task)
        ]
        .set_index("Month")
        [
            [
                "ActualDelayRate",
                "SeasonalOriginRate",
                "SeasonalAirlineRate",
            ]
        ]
    )

    ax = plot_data.plot(
        figsize=(13, 5),
        marker="o",
    )

    ax.axvline(
        x=11.5,
        color="gray",
        linestyle="--",
        label="Train/Validation Boundary",
    )

    ax.set_title(
        f"{task}: Actual vs. Seasonal Historical Rates"
    )

    ax.set_ylabel("Delay Rate")
    ax.set_xlabel("Month")
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
diagnostics = seasonal_monthly_diagnostics.copy()

for group in ["Origin", "Airline"]:

    diagnostics[f"{group}Gap"] = (
        diagnostics[f"Seasonal{group}Rate"]
        - diagnostics["ActualDelayRate"]
    )

    diagnostics[f"{group}AbsoluteGap"] = (
        diagnostics[f"{group}Gap"].abs()
    )

    diagnostics[f"{group}SquaredGap"] = (
        diagnostics[f"{group}Gap"] ** 2
    )

seasonal_drift_summary = (
    diagnostics.groupby(["Split", "Task"])
    .agg(
        Months=("Month", "nunique"),
        OriginBias=("OriginGap", "mean"),
        AirlineBias=("AirlineGap", "mean"),
        OriginMAE=("OriginAbsoluteGap", "mean"),
        AirlineMAE=("AirlineAbsoluteGap", "mean"),
        OriginRMSE=("OriginSquaredGap", "mean"),
        AirlineRMSE=("AirlineSquaredGap", "mean"),
    )
    .reset_index()
)

for column in ["OriginRMSE", "AirlineRMSE"]:
    seasonal_drift_summary[column] = np.sqrt(
        seasonal_drift_summary[column]
    )

print("=== Seasonal Historical Drift Summary ===")

display(seasonal_drift_summary.round(6))

In [ ]:
V2_DIR = ROOT / "data/interim/historical_features_v1"

v2_columns = [
    "origin_dep_delay_rate_3m",
    "origin_outbound_arr_delay_rate_3m",
    "airline_origin_dep_delay_rate_3m",
    "airline_origin_arr_delay_rate_3m",
]

comparison_records = []

for month, v4 in seasonal_features_by_month.items():

    target = load_verified_month(month)

    v2 = pd.read_parquet(
        V2_DIR / f"historical_{month}.parquet"
    )

    for frame in [v2, v4]:

        pd.testing.assert_frame_equal(
            frame[IDENTITY_COLUMNS].reset_index(drop=True),
            target[IDENTITY_COLUMNS].reset_index(drop=True),
            check_dtype=False,
        )

    v2 = v2.reset_index(drop=True)
    v4 = v4.reset_index(drop=True)
    target = target.reset_index(drop=True)

    # Compare exactly the same flight records.
    common = (
        v2[v2_columns].notna().all(axis=1)
        & v4[list(SEASONAL_FEATURE_COLUMNS)]
        .notna().all(axis=1)
    )

    for task, label, offset in [
        ("Departure", "DepDel15", 0),
        ("Arrival", "ArrDel15", 1),
    ]:
        actual = target.loc[common, label].mean()

        for group, position in [
            ("Origin", offset),
            ("Airline", offset + 2),
        ]:
            v2_rate = v2.loc[
                common, v2_columns[position]
            ].mean()

            v4_rate = v4.loc[
                common, SEASONAL_FEATURE_COLUMNS[position]
            ].mean()

            comparison_records.append({
                "Month": month,
                "Task": task,
                "Group": group,
                "Records": int(common.sum()),
                "ActualRate": actual,
                "V2Rate": v2_rate,
                "V4Rate": v4_rate,
                "V2AbsoluteGap": abs(v2_rate - actual),
                "V4AbsoluteGap": abs(v4_rate - actual),
            })

    del target, v2, v4

seasonal_vs_v2 = pd.DataFrame(
    comparison_records
)

seasonal_vs_v2["Split"] = np.where(
    seasonal_vs_v2["Month"].str.startswith("2023"),
    "Training",
    "Validation",
)

seasonal_vs_v2["V4Improved"] = (
    seasonal_vs_v2["V4AbsoluteGap"]
    < seasonal_vs_v2["V2AbsoluteGap"]
)

seasonal_comparison_summary = (
    seasonal_vs_v2
    .groupby(["Split", "Task", "Group"])
    .agg(
        Months=("Month", "nunique"),
        MonthsImproved=("V4Improved", "sum"),
        V2_MAE=("V2AbsoluteGap", "mean"),
        V4_MAE=("V4AbsoluteGap", "mean"),
    )
    .reset_index()
)

seasonal_comparison_summary["MAEReductionPct"] = (
    1
    - seasonal_comparison_summary["V4_MAE"]
    / seasonal_comparison_summary["V2_MAE"]
) * 100

display(seasonal_comparison_summary.round(6))

In [ ]:
validation_drift = (
    seasonal_comparison_summary.loc[
        seasonal_comparison_summary["Split"].eq(
            "Validation"
        )
    ]
    .set_index(["Task", "Group"])
)

ax = validation_drift[
    ["V2_MAE", "V4_MAE"]
].plot.bar(
    figsize=(10, 5),
    rot=0,
)

ax.set_title(
    "Seasonal Matching: Monthly Rate Mismatch"
)

ax.set_ylabel("Mean Absolute Error")
ax.grid(axis="y", alpha=0.3)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.4f",
        padding=3,
    )

plt.tight_layout()
plt.show()

## 5. Seasonal Feature Persistence

Persist monthly seasonal historical features with
candidate flight identity keys and source provenance.

Record feature configuration, source checksums,
target cohort checksums, and generated feature checksums.

Verify all saved feature tables through Parquet
round-trip inspection.

Feature version: seasonal_history_v1

In [ ]:
SEASONAL_DIR = ROOT / "data/interim/seasonal_history_v1"

SEASONAL_DIR.mkdir(parents=True, exist_ok=True)

seasonal_manifest = {
    "feature_version": "seasonal_history_v1",
    "feature_columns": list(SEASONAL_FEATURE_COLUMNS),
    "identity_columns": IDENTITY_COLUMNS,
    "policy": {
        "source_window": "previous_year_same_month",
        "availability_policy": "research_proxy",
        "availability_evidence": "assumed_not_verified",
        "lag_days": LAG_DAYS,
        "min_airline_flights": MIN_AIRLINE_FLIGHTS,
        "fallback_strategy": "origin_level",
    },
    "months": {},
}

for month, frame in seasonal_features_by_month.items():

    path = SEASONAL_DIR / f"seasonal_{month}.parquet"

    frame.to_parquet(path, index=False)

    restored = pd.read_parquet(path)

    pd.testing.assert_frame_equal(
        restored,
        frame.reset_index(drop=True),
    )

    source_month = str(
        pd.Period(month, freq="M") - 12
    )

    with (MANIFEST_DIR / f"{month}.json").open() as f:
        target_manifest = json.load(f)

    with (
        MANIFEST_DIR / f"{source_month}.json"
    ).open() as f:
        source_manifest = json.load(f)

    seasonal_manifest["months"][month] = {
        "filename": path.name,
        "row_count": len(frame),
        "source_month": source_month,
        "sha256": sha256_file(path),
        "target_processed_sha256": (
            target_manifest["processed"]["sha256"]
        ),
        "source_processed_sha256": (
            source_manifest["processed"]["sha256"]
        ),
        "complete_rows": int(
            frame[list(SEASONAL_FEATURE_COLUMNS)]
            .notna().all(axis=1).sum()
        ),
    }

    print(f"{month} | rows={len(frame):,} | verified")

    del restored

with (
    SEASONAL_DIR / "feature_manifest.json"
).open("w", encoding="utf-8") as f:

    json.dump(seasonal_manifest, f, indent=2)

assert len(seasonal_manifest["months"]) == 20

print("\nSeasonal features saved and verified!")

In [ ]:
SEASONAL_ARTIFACTS = ROOT / "artifacts/seasonal_history"

SEASONAL_ARTIFACTS.mkdir(
    parents=True,
    exist_ok=True,
)

tables = {
    "feature_coverage": seasonal_audit,
    "monthly_diagnostics": seasonal_monthly_diagnostics,
    "drift_summary": seasonal_drift_summary,
    "v2_v4_comparison": seasonal_comparison_summary,
    "monthly_v2_v4_comparison": seasonal_vs_v2,
}

for name, table in tables.items():

    table.to_csv(
        SEASONAL_ARTIFACTS / f"{name}.csv",
        index=False,
    )

print("Seasonal diagnostic artifacts saved!")